- Historia 36 meses para todos los productos completados con ceros

In [1]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
warnings.filterwarnings("ignore")

In [2]:
sys.path.append('../../notebooks/entregable/scripts')
import dataset
import preprocesamiento
import target
import feature_engineering
importlib.reload(dataset)
importlib.reload(preprocesamiento)
importlib.reload(target)
importlib.reload(feature_engineering)

Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


<module 'feature_engineering' from 'c:\\Users\\Usuario\\Documents\\Universidad\\austral\\2025\\Lab3\\Lab3-MCD\\notebooks\\model_arima\\../../notebooks/entregable/scripts\\feature_engineering.py'>

In [3]:
df = pd.read_csv("../../data/preprocessed/base.csv", sep=',')
df.shape

(2945818, 13)

In [4]:
#### COMBINATORIA ####
data = dataset.combinatoria_periodo_producto()
data['periodo'] = data['periodo'].dt.year * 100 + data['periodo'].dt.month
data.shape

(44388, 2)

In [5]:
productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")
data = data[data['product_id'].isin(productos_ok['product_id'].unique())]
data

,product_id,periodo
0,20524,201701
1,20524,201702
2,20524,201703
3,20524,201704
4,20524,201705
...,...,...
42835,20127,201908
42836,20127,201909
42837,20127,201910
42838,20127,201911


In [6]:
#### MERGE CON PRODUCTOS ####
productos = pd.read_csv("../../data/raw/tb_productos.csv", sep='\t')
productos = productos.drop_duplicates(subset=['product_id'], keep='first')
data = data.merge(productos, how='left', on="product_id")
del productos

#### MERGE CON STOCKS ####
stocks = pd.read_csv("../../data/raw/tb_stocks.csv", sep='\t')
stocks = stocks.groupby(by=["periodo", "product_id"]).agg({"stock_final": "sum"}).reset_index()
data = data.merge(stocks, how='left', on=['periodo', 'product_id'])
del stocks

#### MERGE CON SELLIN ####
sellin = pd.read_csv("../../data/raw/sell-in.csv", sep='\t')
sellin = sellin.groupby(by=["periodo","product_id"]).agg({"tn":"sum", "plan_precios_cuidados":"sum", "cust_request_qty":"sum", "cust_request_tn":"sum"}).reset_index()
data = data.merge(sellin, how='left', on=['periodo', 'product_id'])
del sellin
gc.collect()

20

In [7]:
#### COMPLETO TN CON CEROS ####
####  ¿cuantos?
print(f"Total de periodos con Nan debido a la combinatoria periodo_x_producto: {data['tn'].isna().sum()}")
#### Lo completo con ceros
data['tn'] = data['tn'].fillna(0)

Total de periodos con Nan debido a la combinatoria periodo_x_producto: 5731


In [8]:
#### GUARDAR DATAFRAME ####
data.to_csv("./datasets/periodo_x_producto.csv", index=False, sep=',', encoding='utf-8')

In [9]:
data = pd.read_csv("./datasets/periodo_x_producto.csv", sep=',')

In [10]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, Naive, SeasonalNaive, HoltWinters
from joblib import Parallel, delayed  # Para paralelizar (opcional)

In [11]:
# Creo DF
ts = data[['product_id','periodo', 'tn']].copy()

# Convertir 'periodo' (yyyymm) a datetime
ts['periodo_dt'] = pd.to_datetime(ts['periodo'].astype(str), format='%Y%m')
ts = ts.sort_values(['product_id', 'periodo_dt'])  # Ordenar por producto y fecha

ts.drop(columns=['periodo'], inplace=True)  # Eliminar columna temporal
ts.rename(columns={'tn': 'y', 'periodo_dt':'ds', 'product_id':'unique_id'}, inplace=True)  # Renombrar columna de tn

In [12]:
models = [
    AutoARIMA(season_length=12),
    AutoETS(season_length=12),
    AutoTheta(season_length=12)
]


predictions = {}

for product_id in productos_ok['product_id'].unique():
    df_product = ts[ts['unique_id'] == product_id].copy()

    # Añadir columna unique_id (requerida por StatsForecast)
    df_product['unique_id'] = product_id

    # Seleccionar columnas necesarias y ordenar por fecha
    df_product = df_product[['unique_id', 'ds', 'y']].sort_values('ds')

    if len(df_product) >= 12:
        try:
            sf = StatsForecast(models=models, freq='MS', n_jobs=-1)
            pred = sf.forecast(h=2, df=df_product)

            # Obtener la última predicción (h=2)
            pred_feb2020 = pred.sort_values('ds').iloc[[-1]]

            # Calcular promedio de modelos
            mean_pred = pred_feb2020[['AutoARIMA', 'AutoETS', 'AutoTheta']].mean(axis=1).iloc[0]
            predictions[product_id] = mean_pred

            print(f"Producto {product_id}: Modelo ajustado")

        except Exception as e:
            print(f"Error en producto {product_id}: {str(e)}")
            predictions[product_id] = None
    else:
        print(f"Producto {product_id}: Insuficientes datos ({len(df_product)} observaciones)")
        predictions[product_id] = None

# Convertir a DataFrame
df_predictions = pd.DataFrame({
    'product_id': predictions.keys(),
    'prediccion_mes+2': predictions.values()
})

Producto 20001: Modelo ajustado
Producto 20002: Modelo ajustado
Producto 20003: Modelo ajustado
Producto 20004: Modelo ajustado
Producto 20005: Modelo ajustado
Producto 20006: Modelo ajustado
Producto 20007: Modelo ajustado
Producto 20008: Modelo ajustado
Producto 20009: Modelo ajustado
Producto 20010: Modelo ajustado
Producto 20011: Modelo ajustado
Producto 20012: Modelo ajustado
Producto 20013: Modelo ajustado
Producto 20014: Modelo ajustado
Producto 20015: Modelo ajustado
Producto 20016: Modelo ajustado
Producto 20017: Modelo ajustado
Producto 20018: Modelo ajustado
Producto 20019: Modelo ajustado
Producto 20020: Modelo ajustado
Producto 20021: Modelo ajustado
Producto 20022: Modelo ajustado
Producto 20023: Modelo ajustado
Producto 20024: Modelo ajustado
Producto 20025: Modelo ajustado
Producto 20026: Modelo ajustado
Producto 20027: Modelo ajustado
Producto 20028: Modelo ajustado
Producto 20029: Modelo ajustado
Producto 20030: Modelo ajustado
Producto 20031: Modelo ajustado
Producto

In [13]:
def promedio_12_meses_780p():
    
    df = pd.read_csv("./datasets/periodo_x_producto_con_target.csv", sep=',', encoding='utf-8')
    df = df[df['periodo'] >= 201901]  # Filtrar desde 201901
    
    productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")

    df = df.merge(productos_ok, on='product_id', how='inner')
    
    df = df.groupby('product_id').agg({'tn': 'mean'}).reset_index()
    
    return df

df_promedios = promedio_12_meses_780p()


In [ ]:
df_predictions = df_predictions.merge(df_promedios, on='product_id', how='left')
df_predictions

In [ ]:
df_predictions.loc[df_predictions['tn_x'] < 0, 'tn_x'] = df_predictions['tn_y']
df_predictions.rename(columns={'tn_x': 'tn'}, inplace=True)
df_predictions[['product_id', 'tn']].to_csv("./datasets/autoarima_exp03.csv", index=False, sep=',', encoding='utf-8')

In [17]:
df_predictions.rename(columns={'prediccion_mes+2': 'tn'}, inplace=True)
df_predictions[['product_id', 'tn']].to_csv("./datasets/autoarima_exp03.csv", index=False, sep=',', encoding='utf-8')